# DataCo Smart Supply Chain Analysis

This notebook analyzes the DataCo Smart Supply Chain dataset using Python, focusing on data quality, cleaning, business performance, demand forecasting, inventory planning, logistics, profitability, and preparation for SQL and Power BI.

**Dataset grain:** The source is at the order-item level, so order-level metrics use distinct `Order Id` values where appropriate.

**Key limitations:** Supplier and actual inventory data are not available; inventory planning is therefore derived. Historical shipping duration is used as a lead-time proxy, EOQ uses assumed ordering and holding costs, and 2018 contains January data only.

## 1. Data Loading

This section imports the required Python libraries and loads the DataCo Smart Supply Chain CSV dataset for analysis.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../Datasets/raw/DataCoSupplyChainDataset.csv", encoding="latin1")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.info()

Rows: 180519
Columns: 53
<class 'pandas.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  str    
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  str    
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  str    
 9   Customer City                  180519 non-null  str    
 10  Customer Country               180519 non-null  str    
 11  Customer Email                 180519 non-null  str    
 12  Customer Fname  

## 2. Data Quality Audit

This section checks the dataset structure, missing values, duplicate records, and basic data quality issues before cleaning.

In [456]:
missing = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing %": (df.isna().sum() / len(df) * 100).round(2)
})

missing = missing[missing["Missing Values"] > 0]

missing.sort_values("Missing Values", ascending=False)


,Missing Values,Missing %
Product Description,180519,100.00
Order Zipcode,155679,86.24
Customer Lname,8,0.00
Customer Zipcode,3,0.00


In [457]:
print("Duplicate rows:", df.duplicated().sum())
print("Unique Order IDs:", df["Order Id"].nunique())
print("Total rows:", len(df))

Duplicate rows: 0
Unique Order IDs: 65752
Total rows: 180519


In [458]:
order_counts = df["Order Id"].value_counts()
order_counts.head(10)


Order Id
45461    5
31115    5
47752    5
17162    5
64813    5
12525    5
5895     5
6783     5
56973    5
5991     5
Name: count, dtype: int64

In [459]:
order_counts.value_counts().sort_index()

count
1    19850
2    11447
3    11398
4    11704
5    11353
Name: count, dtype: int64

In [460]:
example_order = order_counts[order_counts > 1].index[0]

df[df["Order Id"] == example_order][
    [
        "Order Id",
        "Order Item Id",
        "Product Card Id",
        "Product Name",
        "Order Item Quantity",
        "Sales",
        "Order Item Total"
    ]
]

,Order Id,Order Item Id,Product Card Id,Product Name,Order Item Quantity,Sales,Order Item Total
49,45461,113598,627,Under Armour Girls' Toddler Spine Surge Runni,2,79.980003,79.180000
515,45461,113596,502,Nike Men's Dri-FIT Victory Golf Polo,4,200.000000,164.000000
112661,45461,113600,1014,O'Brien Men's Neoprene Life Vest,3,149.940002,119.949997
179242,45461,113597,403,Nike Men's CJ Elite 2 TD Football Cleat,1,129.990005,97.489998
179305,45461,113599,191,Nike Men's Free 5.0+ Running Shoe,2,199.979996,159.979996


In [461]:
print("Unique Order IDs:", df["Order Id"].nunique())
print("Unique Order Item IDs:", df["Order Item Id"].nunique())
print("Duplicate Order Item IDs:", df["Order Item Id"].duplicated().sum())

Unique Order IDs: 65752
Unique Order Item IDs: 180519
Duplicate Order Item IDs: 0


In [462]:
print("Unique Customer Id:", df["Customer Id"].nunique())
print("Unique Order Customer Id:", df["Order Customer Id"].nunique())

print(
    "Rows where Customer Id != Order Customer Id:",
    (df["Customer Id"] != df["Order Customer Id"]).sum()
)

Unique Customer Id: 20652
Unique Order Customer Id: 20652
Rows where Customer Id != Order Customer Id: 0


In [463]:
print("Unique Product Card IDs:", df["Product Card Id"].nunique())
print("Unique Order Item Cardprod IDs:", df["Order Item Cardprod Id"].nunique())

print(
    "Rows where Product Card Id != Order Item Cardprod Id:",
    (df["Product Card Id"] != df["Order Item Cardprod Id"]).sum()
)

Unique Product Card IDs: 118
Unique Order Item Cardprod IDs: 118
Rows where Product Card Id != Order Item Cardprod Id: 0


In [464]:
product_name_check = (
    df.groupby("Product Card Id")["Product Name"]
      .nunique()
)

print("Products with multiple names:", (product_name_check > 1).sum())
print("Maximum names for one product:", product_name_check.max())

Products with multiple names: 0
Maximum names for one product: 1


In [465]:
product_check = (
    df.groupby("Product Card Id")
      .agg({
          "Product Name": "nunique",
          "Category Name": "nunique",
          "Department Name": "nunique",
          "Order Item Product Price": "nunique"
      })
)

print("Products with multiple categories:",
      (product_check["Category Name"] > 1).sum())

print("Products with multiple departments:",
      (product_check["Department Name"] > 1).sum())

print("Products with multiple prices:",
      (product_check["Order Item Product Price"] > 1).sum())

Products with multiple categories: 0
Products with multiple departments: 0
Products with multiple prices: 0


## 3. Detailed Data Validation

This section validates dates, shipping information, and product attributes to identify inconsistencies before cleaning.

In [466]:
print("Order date data type:", df["order date (DateOrders)"].dtype)
print("Shipping date data type:", df["shipping date (DateOrders)"].dtype)

print("\nFirst 5 order dates:")
print(df["order date (DateOrders)"].head())

print("\nFirst 5 shipping dates:")
print(df["shipping date (DateOrders)"].head())
print("Order date range:")
print(df["order date (DateOrders)"].min(), "to", df["order date (DateOrders)"].max())

print("\nShipping date range:")
print(df["shipping date (DateOrders)"].min(), "to", df["shipping date (DateOrders)"].max())

Order date data type: str
Shipping date data type: str

First 5 order dates:
0    1/31/2018 22:56
1    1/13/2018 12:27
2    1/13/2018 12:06
3    1/13/2018 11:45
4    1/13/2018 11:24
Name: order date (DateOrders), dtype: str

First 5 shipping dates:
0     2/3/2018 22:56
1    1/18/2018 12:27
2    1/17/2018 12:06
3    1/16/2018 11:45
4    1/15/2018 11:24
Name: shipping date (DateOrders), dtype: str
Order date range:
1/1/2015 0:00 to 9/9/2017 9:50

Shipping date range:
1/1/2016 0:22 to 9/9/2017 9:08


In [467]:
order_dates = pd.to_datetime(
    df["order date (DateOrders)"],
    format="%m/%d/%Y %H:%M"
)

shipping_dates = pd.to_datetime(
    df["shipping date (DateOrders)"],
    format="%m/%d/%Y %H:%M"
)

print("Actual order date range:")
print(order_dates.min(), "to", order_dates.max())

print("\nActual shipping date range:")
print(shipping_dates.min(), "to", shipping_dates.max())

Actual order date range:
2015-01-01 00:00:00 to 2018-01-31 23:38:00

Actual shipping date range:
2015-01-03 00:00:00 to 2018-02-06 22:14:00


In [468]:
print("Actual shipping days:")
print(df["Days for shipping (real)"].describe())

print("\nScheduled shipping days:")
print(df["Days for shipment (scheduled)"].describe())

print("\nNegative actual shipping days:",
      (df["Days for shipping (real)"] < 0).sum())

print("Negative scheduled shipping days:",
      (df["Days for shipment (scheduled)"] < 0).sum())

Actual shipping days:
count    180519.000000
mean          3.497654
std           1.623722
min           0.000000
25%           2.000000
50%           3.000000
75%           5.000000
max           6.000000
Name: Days for shipping (real), dtype: float64

Scheduled shipping days:
count    180519.000000
mean          2.931847
std           1.374449
min           0.000000
25%           2.000000
50%           4.000000
75%           4.000000
max           4.000000
Name: Days for shipment (scheduled), dtype: float64

Negative actual shipping days: 0
Negative scheduled shipping days: 0


In [469]:
print("Delivery Status values:")
print(df["Delivery Status"].value_counts())

print("\nLate delivery risk:")
print(df["Late_delivery_risk"].value_counts())

print("\nShipping Mode:")
print(df["Shipping Mode"].value_counts())

Delivery Status values:
Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

Late delivery risk:
Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

Shipping Mode:
Shipping Mode
Standard Class    107752
Second Class       35216
First Class        27814
Same Day            9737
Name: count, dtype: int64


## 4. Financial & Transaction Validation

This section validates the financial fields and transaction-level relationships in the dataset.

The checks focus on:
- Sales and order-item quantities
- Order-item totals
- Product prices
- Discounts
- Profit
- Relationships between financial fields
- Negative or invalid financial values

In [470]:
financial_cols = [
    "Sales",
    "Order Item Total",
    "Order Item Product Price",
    "Order Item Quantity",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Profit Ratio",
    "Benefit per order"
]

print(df[financial_cols].describe().T)
print("\nNegative values:")

for col in financial_cols:
    print(col, ":", (df[col] < 0).sum())

                             count        mean         std         min  \
Sales                     180519.0  203.772096  132.273077     9.99000   
Order Item Total          180519.0  183.107609  120.043670     7.49000   
Order Item Product Price  180519.0  141.232550  139.732492     9.99000   
Order Item Quantity       180519.0    2.127638    1.453451     1.00000   
Order Item Discount       180519.0   20.664741   21.800901     0.00000   
Order Item Discount Rate  180519.0    0.101668    0.070415     0.00000   
Order Item Profit Ratio   180519.0    0.120647    0.466796    -2.75000   
Benefit per order         180519.0   21.974989  104.433526 -4274.97998   

                                 25%         50%         75%          max  
Sales                     119.980003  199.919998  299.950012  1999.989990  
Order Item Total          104.379997  163.990005  247.399994  1939.989990  
Order Item Product Price   50.000000   59.990002  199.990005  1999.989990  
Order Item Quantity         1

In [471]:
df["Calculated Total"] = (
    df["Order Item Product Price"] * df["Order Item Quantity"]
    - df["Order Item Discount"]
)

df["Total Difference"] = (
    df["Calculated Total"] - df["Order Item Total"]
)

print(df["Total Difference"].describe())
print(
    "Rows with calculation difference > 0.01:",
    (df["Total Difference"].abs() > 0.01).sum()
)

count    180519.000000
mean         -0.000254
std           0.001577
min          -0.010025
25%           0.000000
50%           0.000000
75%           0.000004
max           0.000042
Name: Total Difference, dtype: float64
Rows with calculation difference > 0.01: 1446


In [472]:
financial_mismatch = df[
    df["Total Difference"].abs() > 0.01
]

financial_mismatch[
    [
        "Order Item Id",
        "Product Card Id",
        "Order Item Product Price",
        "Order Item Quantity",
        "Order Item Discount",
        "Calculated Total",
        "Order Item Total",
        "Total Difference"
    ]
].head(10)

,Order Item Id,Product Card Id,Order Item Product Price,Order Item Quantity,Order Item Discount,Calculated Total,Order Item Total,Total Difference
5,179250,1360,327.750000,1,32.779999,294.970001,294.980011,-0.010010
11,179244,1360,327.750000,1,59.000000,268.750000,268.760010,-0.010010
16,179239,1360,327.750000,1,6.560000,321.190000,321.200012,-0.010012
23,179232,1360,327.750000,1,32.779999,294.970001,294.980011,-0.010010
29,179226,1360,327.750000,1,59.000000,268.750000,268.760010,-0.010010
34,179221,1360,327.750000,1,6.560000,321.190000,321.200012,-0.010012
41,179214,1360,327.750000,1,32.779999,294.970001,294.980011,-0.010010
47,179208,1360,327.750000,1,59.000000,268.750000,268.760010,-0.010010
127,143043,191,99.989998,5,50.000000,449.949989,449.959992,-0.010002
128,21551,191,99.989998,5,50.000000,449.949989,449.959992,-0.010002


In [473]:
df.drop(
    columns=["Calculated Total", "Total Difference"],
    inplace=True
)
print(df.shape)

(180519, 53)


## 5. Data Cleaning

We create `clean_df` so the raw dataset remains unchanged.

Columns that are sensitive, redundant, incomplete, or unsuitable for the analytical model are removed while retaining fields required for downstream analysis.The order and shipping date columns are converted from text to datetime format.


In [474]:
clean_df = df.copy()

print(clean_df.shape)

(180519, 53)


In [475]:
columns_to_drop = [
    "Product Description",
    "Customer Password",
    "Customer Email",
    "Product Image",
    "Order Zipcode",
    "Order Customer Id",
    "Order Item Cardprod Id"
]

clean_df.drop(columns=columns_to_drop, inplace=True)

print(clean_df.shape)

(180519, 46)


In [476]:
clean_df["order date (DateOrders)"] = pd.to_datetime(
    clean_df["order date (DateOrders)"],
    format="%m/%d/%Y %H:%M"
)

clean_df["shipping date (DateOrders)"] = pd.to_datetime(
    clean_df["shipping date (DateOrders)"],
    format="%m/%d/%Y %H:%M"
)

print(clean_df[[
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]].dtypes)

order date (DateOrders)       datetime64[us]
shipping date (DateOrders)    datetime64[us]
dtype: object


In [477]:
print("Rows:", len(clean_df))
print("Columns:", len(clean_df.columns))
print("Duplicate rows:", clean_df.duplicated().sum())
print("Missing values:", clean_df.isnull().sum().sum())

Rows: 180519
Columns: 46
Duplicate rows: 0
Missing values: 11


In [478]:
clean_df["Customer Zipcode"] = clean_df["Customer Zipcode"].fillna("Unknown")

print("Missing values:", clean_df.isnull().sum().sum())

Missing values: 8


In [479]:
categorical_columns = [
    "Market",
    "Order Region",
    "Customer Segment",
    "Shipping Mode",
    "Delivery Status",
    "Category Name",
    "Department Name"
]

for col in categorical_columns:
    print(f"\n--- {col} ---")
    print(clean_df[col].value_counts())


--- Market ---
Market
LATAM           51594
Europe          50252
Pacific Asia    41260
USCA            25799
Africa          11614
Name: count, dtype: int64

--- Order Region ---
Order Region
Central America    28341
Western Europe     27109
South America      14935
Oceania            10148
Northern Europe     9792
Southeast Asia      9539
Southern Europe     9431
Caribbean           8318
West of USA         7993
South Asia          7731
Eastern Asia        7280
East of USA         6915
West Asia           6009
US Center           5887
South of  USA       4045
Eastern Europe      3920
West Africa         3696
North Africa        3232
East Africa         1852
Central Africa      1677
Southern Africa     1157
Canada               959
Central Asia         553
Name: count, dtype: int64

--- Customer Segment ---
Customer Segment
Consumer       93504
Corporate      54789
Home Office    32226
Name: count, dtype: int64

--- Shipping Mode ---
Shipping Mode
Standard Class    107752
Second Clas

## 6. Date Conversion & Feature Engineering

Additional date-based features are created to support yearly, monthly, quarterly, and day-of-week analysis.

Shipping delay and profit margin features are also calculated for logistics and profitability analysis.

In [480]:
clean_df["Order Year"] = clean_df["order date (DateOrders)"].dt.year
clean_df["Order Month"] = clean_df["order date (DateOrders)"].dt.month
clean_df["Order Month Name"] = clean_df["order date (DateOrders)"].dt.month_name()
clean_df["Order Quarter"] = clean_df["order date (DateOrders)"].dt.quarter
clean_df["Order Day"] = clean_df["order date (DateOrders)"].dt.day
clean_df["Order Day of Week"] = clean_df["order date (DateOrders)"].dt.day_name()

print(clean_df[[
    "order date (DateOrders)",
    "Order Year",
    "Order Month",
    "Order Month Name",
    "Order Quarter",
    "Order Day",
    "Order Day of Week"
]].head())

  order date (DateOrders)  Order Year  Order Month Order Month Name  \
0     2018-01-31 22:56:00        2018            1          January   
1     2018-01-13 12:27:00        2018            1          January   
2     2018-01-13 12:06:00        2018            1          January   
3     2018-01-13 11:45:00        2018            1          January   
4     2018-01-13 11:24:00        2018            1          January   

   Order Quarter  Order Day Order Day of Week  
0              1         31         Wednesday  
1              1         13          Saturday  
2              1         13          Saturday  
3              1         13          Saturday  
4              1         13          Saturday  


In [481]:
clean_df["Shipping Delay Days"] = (
    clean_df["Days for shipping (real)"]
    - clean_df["Days for shipment (scheduled)"]
)

print(clean_df["Shipping Delay Days"].describe())

count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: Shipping Delay Days, dtype: float64


In [482]:
clean_df["Order Year-Month"] = (
    clean_df["order date (DateOrders)"]
    .dt.to_period("M")
    .astype(str)
)

print(clean_df[[
    "order date (DateOrders)",
    "Order Year-Month"
]].head())  

  order date (DateOrders) Order Year-Month
0     2018-01-31 22:56:00          2018-01
1     2018-01-13 12:27:00          2018-01
2     2018-01-13 12:06:00          2018-01
3     2018-01-13 11:45:00          2018-01
4     2018-01-13 11:24:00          2018-01


In [483]:
clean_df["Order Day of Week Number"] = (
    clean_df["order date (DateOrders)"].dt.dayofweek + 1
)

print(
    clean_df[
        ["Order Day of Week", "Order Day of Week Number"]
    ].drop_duplicates().sort_values("Order Day of Week Number")
)

   Order Day of Week  Order Day of Week Number
52            Monday                         1
49           Tuesday                         2
0          Wednesday                         3
62          Thursday                         4
37            Friday                         5
1           Saturday                         6
51            Sunday                         7


In [484]:
clean_df["Order Date"] = clean_df["order date (DateOrders)"].dt.date

print(clean_df[[
    "order date (DateOrders)",
    "Order Date"
]].head())

  order date (DateOrders)  Order Date
0     2018-01-31 22:56:00  2018-01-31
1     2018-01-13 12:27:00  2018-01-13
2     2018-01-13 12:06:00  2018-01-13
3     2018-01-13 11:45:00  2018-01-13
4     2018-01-13 11:24:00  2018-01-13


In [485]:
clean_df["Profit Margin (%)"] = (
    clean_df["Benefit per order"] / clean_df["Sales"] * 100
)

print(clean_df["Profit Margin (%)"].describe())

count    180519.000000
mean         10.832612
std          42.059372
min        -274.999998
25%           6.224000
50%          24.251212
75%          33.601431
max          50.044288
Name: Profit Margin (%), dtype: float64


## 7. Final Cleaning Validation

Before beginning the business analysis, the cleaned dataset is validated again.

The validation confirms:

* Row count is unchanged after cleaning.
* Expected feature engineering columns are present.
* No duplicate rows were introduced.
* Remaining missing values are limited to the intentionally retained `Customer Lname` values.


In [486]:
print("Rows:", len(clean_df))
print("Columns:", len(clean_df.columns))
print("Duplicate rows:", clean_df.duplicated().sum())
print("Missing values:", clean_df.isnull().sum().sum())

print("\nNew features:")
print([
    "Order Year",
    "Order Month",
    "Order Month Name",
    "Order Quarter",
    "Order Day",
    "Order Day of Week",
    "Order Day of Week Number",
    "Shipping Delay Days",
    "Order Year-Month",
    "Order Date",
    "Profit Margin (%)"
])

Rows: 180519
Columns: 57
Duplicate rows: 0
Missing values: 8

New features:
['Order Year', 'Order Month', 'Order Month Name', 'Order Quarter', 'Order Day', 'Order Day of Week', 'Order Day of Week Number', 'Shipping Delay Days', 'Order Year-Month', 'Order Date', 'Profit Margin (%)']


## 8. Overall Business Performance

This section evaluates the overall commercial performance of the supply chain dataset.

Key business metrics are calculated first, followed by analysis of sales and order trends across years, months, markets, regions, and product categories.

### 8.1 Core Business KPIs

The following KPIs provide a high-level view of business performance:

- Total Sales
- Total Orders
- Total Quantity
- Total Profit
- Average Order Value (AOV)
- Average Quantity per Order
- Profit Margin

These metrics establish the baseline for the subsequent performance analysis.

In [487]:
total_sales = clean_df["Sales"].sum()
total_orders = clean_df["Order Id"].nunique()
total_quantity = clean_df["Order Item Quantity"].sum()
total_profit = clean_df["Benefit per order"].sum()

average_order_value = total_sales / total_orders
average_quantity_per_order = total_quantity / total_orders
profit_margin = total_profit / total_sales * 100

print("Total Sales:", round(total_sales, 2))
print("Total Orders:", total_orders)
print("Total Quantity Sold:", total_quantity)
print("Total Profit:", round(total_profit, 2))
print("Average Order Value:", round(average_order_value, 2))
print("Average Quantity per Order:", round(average_quantity_per_order, 2))
print("Overall Profit Margin:", round(profit_margin, 2), "%")

Total Sales: 36784735.01
Total Orders: 65752
Total Quantity Sold: 384079
Total Profit: 3966902.97
Average Order Value: 559.45
Average Quantity per Order: 5.84
Overall Profit Margin: 10.78 %


In [488]:
financial_summary = clean_df[
    [
        "Sales",
        "Order Item Total",
        "Order Item Product Price",
        "Order Item Discount",
        "Benefit per order"
    ]
].sum()

print(financial_summary)

Sales                       3.678474e+07
Order Item Total            3.305440e+07
Order Item Product Price    2.549516e+07
Order Item Discount         3.730378e+06
Benefit per order           3.966903e+06
dtype: float64


### 8.2 Yearly Business Performance

Yearly performance is analyzed using total sales, total profit, and number of unique orders.

Year-over-year sales growth is also calculated to identify changes in business performance across the available years.

The 2018 data represents only January, so it should not be interpreted as a full-year result.

In [489]:
yearly_sales = (
    clean_df.groupby("Order Year")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Benefit per order", "sum"),
        Orders=("Order Id", "nunique")
    )
    .reset_index()
)

print(yearly_sales)

   Order Year         Sales        Profit  Orders
0        2015  1.234083e+07  1.318857e+06   20904
1        2016  1.230382e+07  1.310119e+06   20859
2        2017  1.180844e+07  1.304085e+06   21866
3        2018  3.316501e+05  3.384189e+04    2123


In [490]:
yearly_sales["Sales Growth (%)"] = (
    yearly_sales["Sales"].pct_change() * 100
)

print(yearly_sales)

   Order Year         Sales        Profit  Orders  Sales Growth (%)
0        2015  1.234083e+07  1.318857e+06   20904               NaN
1        2016  1.230382e+07  1.310119e+06   20859         -0.299932
2        2017  1.180844e+07  1.304085e+06   21866         -4.026240
3        2018  3.316501e+05  3.384189e+04    2123        -97.191414


### 8.3 Monthly Business Performance

Monthly performance is analyzed to identify changes in sales, profitability, order volume, and quantity over time.

The analysis uses `Order Year-Month` so that the same calendar month from different years is kept as a separate observation.

In [491]:
monthly_sales = (
    clean_df.groupby("Order Year-Month")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Benefit per order", "sum"),
        Orders=("Order Id", "nunique"),
        Quantity=("Order Item Quantity", "sum")
    )
    .reset_index()
)

print(monthly_sales.head(15))

   Order Year-Month         Sales         Profit  Orders  Quantity
0           2015-01  1.051590e+06  111660.740132    1787     11854
1           2015-02  9.270099e+05   99140.660196    1585     10438
2           2015-03  1.051254e+06  113778.210191    1781     12062
3           2015-04  1.014463e+06  108083.679957    1710     11287
4           2015-05  1.050478e+06  112147.900143    1776     11902
5           2015-06  1.024006e+06  110147.160313    1725     11203
6           2015-07  1.038081e+06  115624.059879    1763     11800
7           2015-08  1.029495e+06  117979.770302    1762     11612
8           2015-09  1.018339e+06  113467.940118    1706     11366
9           2015-10  1.049154e+06  101757.870040    1775     11703
10          2015-11  1.029120e+06  104427.360120    1729     11463
11          2015-12  1.057841e+06  110641.549881    1805     11790
12          2016-01  1.046308e+06  106780.950229    1772     11597
13          2016-02  9.685428e+05   86809.490642    1650     1

In [492]:
print("Top 10 months by Sales:")
print(
    monthly_sales
    .sort_values("Sales", ascending=False)
    .head(10)
    [["Order Year-Month", "Sales", "Profit", "Orders"]]
)

print("\nBottom 10 months by Sales:")
print(
    monthly_sales
    .sort_values("Sales", ascending=True)
    .head(10)
    [["Order Year-Month", "Sales", "Profit", "Orders"]]
)

Top 10 months by Sales:
   Order Year-Month         Sales         Profit  Orders
32          2017-09  1.143775e+06  122462.390153    1723
31          2017-08  1.109337e+06  131501.160211    1768
28          2017-05  1.105485e+06  115014.640014    1763
30          2017-07  1.104373e+06  113026.700038    1776
33          2017-10  1.073994e+06  113447.169883    2101
11          2015-12  1.057841e+06  110641.549881    1805
0           2015-01  1.051590e+06  111660.740132    1787
2           2015-03  1.051254e+06  113778.210191    1781
4           2015-05  1.050478e+06  112147.900143    1776
9           2015-10  1.049154e+06  101757.870040    1775

Bottom 10 months by Sales:
   Order Year-Month         Sales         Profit  Orders
36          2018-01  3.316501e+05   33841.889977    2123
35          2017-12  5.039108e+05   65837.629745    2124
34          2017-11  6.269144e+05   67791.250205    2055
1           2015-02  9.270099e+05   99140.660196    1585
13          2016-02  9.685428e+05   

In [493]:
monthly_comparison = (
    clean_df
    .groupby(["Order Year", "Order Month"])["Sales"]
    .sum()
    .unstack(level=0)
)

print(monthly_comparison.round(2))

Order Year         2015        2016        2017       2018
Order Month                                               
1            1051590.08  1046308.25  1029698.02  331650.12
2             927009.90   968542.85   992534.90        NaN
3            1051253.69  1025853.12  1048004.78        NaN
4            1014463.28  1001211.58  1038321.62        NaN
5            1050478.44  1029400.20  1105485.32        NaN
6            1024006.17  1003059.55  1032086.49        NaN
7            1038081.19  1045715.62  1104373.36        NaN
8            1029494.69  1048200.25  1109337.17        NaN
9            1018338.60  1002397.04  1143775.11        NaN
10           1049154.27  1048130.55  1073994.17        NaN
11           1029120.24  1047590.14   626914.38        NaN
12           1057840.88  1037408.17   503910.82        NaN


In [494]:
monthly_comparison["2016 vs 2015 (%)"] = (
    (monthly_comparison[2016] - monthly_comparison[2015])
    / monthly_comparison[2015] * 100
)

monthly_comparison["2017 vs 2016 (%)"] = (
    (monthly_comparison[2017] - monthly_comparison[2016])
    / monthly_comparison[2016] * 100
)

print(
    monthly_comparison[
        [
            2015,
            2016,
            2017,
            "2016 vs 2015 (%)",
            "2017 vs 2016 (%)"
        ]
    ].round(2)
)

Order Year         2015        2016        2017  2016 vs 2015 (%)  \
Order Month                                                         
1            1051590.08  1046308.25  1029698.02             -0.50   
2             927009.90   968542.85   992534.90              4.48   
3            1051253.69  1025853.12  1048004.78             -2.42   
4            1014463.28  1001211.58  1038321.62             -1.31   
5            1050478.44  1029400.20  1105485.32             -2.01   
6            1024006.17  1003059.55  1032086.49             -2.05   
7            1038081.19  1045715.62  1104373.36              0.74   
8            1029494.69  1048200.25  1109337.17              1.82   
9            1018338.60  1002397.04  1143775.11             -1.57   
10           1049154.27  1048130.55  1073994.17             -0.10   
11           1029120.24  1047590.14   626914.38              1.79   
12           1057840.88  1037408.17   503910.82             -1.93   

Order Year   2017 vs 2016 (%)  
O

### 8.4 Market & Regional Performance

Market and regional performance is analyzed to understand how sales changed across geographic business segments.

The analysis focuses on November and December 2016 versus 2017, following the original analysis. These months are examined separately rather than combining them across years.

In [495]:
market_monthly = (
    clean_df[
        clean_df["Order Year"].isin([2016, 2017])
    ]
    .groupby(["Market", "Order Year", "Order Month"])["Sales"]
    .sum()
    .reset_index()
)

late_2017 = market_monthly[
    (market_monthly["Order Year"] == 2017) &
    (market_monthly["Order Month"].isin([11, 12]))
]

print(
    late_2017
    .sort_values("Sales", ascending=False)
    .round(2)
)

          Market  Order Year  Order Month      Sales
35  Pacific Asia        2017           12  503910.82
34  Pacific Asia        2017           11  425291.12
17        Europe        2017           11  201623.26


In [496]:
market_comparison = (
    clean_df[
        clean_df["Order Month"].isin([11, 12]) &
        clean_df["Order Year"].isin([2016, 2017])
    ]
    .groupby(["Market", "Order Year", "Order Month"])["Sales"]
    .sum()
    .unstack("Order Year")
    .reset_index()
)

market_comparison["Change (%)"] = (
    (market_comparison[2017] - market_comparison[2016])
    / market_comparison[2016]
    * 100
)

print(
    market_comparison
    .sort_values("Change (%)")
    .round(2)
)

Order Year        Market  Order Month       2016       2017  Change (%)
2                 Europe           11  194659.65  201623.26        3.58
4           Pacific Asia           11  312580.49  425291.12       36.06
5           Pacific Asia           12  328521.19  503910.82       53.39
0                 Africa           11  492548.65        NaN         NaN
1                 Africa           12  487694.12        NaN         NaN
3                 Europe           12  180441.81        NaN         NaN
6                   USCA           11   47801.35        NaN         NaN
7                   USCA           12   40751.05        NaN         NaN


In [497]:
region_comparison = (
    clean_df[
        clean_df["Order Month"].isin([11, 12]) &
        clean_df["Order Year"].isin([2016, 2017])
    ]
    .groupby(
        ["Market", "Order Region", "Order Year", "Order Month"]
    )["Sales"]
    .sum()
    .unstack("Order Year")
    .reset_index()
)

region_comparison["Change (%)"] = (
    (region_comparison[2017] - region_comparison[2016])
    / region_comparison[2016]
    * 100
)

print(
    region_comparison
    .sort_values(["Market", "Order Month", "Change (%)"])
    .round(2)
)

Order Year        Market     Order Region  Order Month       2016       2017  \
0                 Africa   Central Africa           11   87624.81        NaN   
2                 Africa      East Africa           11   79380.78        NaN   
4                 Africa     North Africa           11  141352.82        NaN   
6                 Africa  Southern Africa           11   46113.02        NaN   
8                 Africa      West Africa           11  138077.21        NaN   
1                 Africa   Central Africa           12   64733.09        NaN   
3                 Africa      East Africa           12   88613.12        NaN   
5                 Africa     North Africa           12  119448.83        NaN   
7                 Africa  Southern Africa           12   49460.22        NaN   
9                 Africa      West Africa           12  165438.85        NaN   
14                Europe  Southern Europe           11    7824.03   42350.22   
16                Europe   Western Europ

In [498]:
coverage_check = (
    clean_df[
        clean_df["Order Year"].isin([2016, 2017])
    ]
    .groupby(["Order Year", "Order Month", "Market"])
    .size()
    .reset_index(name="Records")
)

print(
    coverage_check[
        coverage_check["Order Month"].isin([10, 11, 12])
    ]
    .sort_values(["Order Year", "Order Month", "Market"])
    .to_string(index=False)
)

 Order Year  Order Month       Market  Records
       2016           10       Africa     2383
       2016           10       Europe      980
       2016           10 Pacific Asia     1862
       2016           10         USCA      173
       2016           11       Africa     2426
       2016           11       Europe      988
       2016           11 Pacific Asia     1577
       2016           11         USCA      219
       2016           12       Africa     2476
       2016           12       Europe      921
       2016           12 Pacific Asia     1663
       2016           12         USCA      209
       2017           10       Europe     2255
       2017           11       Europe      682
       2017           11 Pacific Asia     1373
       2017           12 Pacific Asia     2124


### 8.5 Category Performance

Category-level performance is analyzed to understand how different product categories contribute to sales, profit, order volume, and quantity sold.

In [499]:
category_analysis = (
    clean_df
    .groupby("Category Name")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Benefit per order", "sum"),
        Quantity=("Order Item Quantity", "sum"),
        Orders=("Order Id", "nunique")
    )
    .reset_index()
)

category_analysis["Profit Margin (%)"] = (
    category_analysis["Profit"]
    / category_analysis["Sales"]
    * 100
)

category_analysis = category_analysis.sort_values(
    "Sales",
    ascending=False
)

print(category_analysis.round(2).to_string(index=False))

       Category Name      Sales    Profit  Quantity  Orders  Profit Margin (%)
             Fishing 6929653.69 756220.77     17325   15164              10.91
              Cleats 4431942.78 494636.92     73734   20386              11.16
    Camping & Hiking 4118425.57 427455.57     13729   12299              10.38
    Cardio Equipment 3694843.20 383011.10     37587   11355              10.37
     Women's Apparel 3147800.00 350421.03     62956   17869              11.13
        Water Sports 3113844.68 325146.96     15540   13758              10.44
      Men's Footwear 2891757.66 311902.82     22246   18783              10.79
Indoor/Outdoor Games 2888993.91 318451.43     57803   16623              11.02
       Shop By Sport 1309522.04 129813.96     32726   10136               9.91
           Computers  663000.00  69656.81       442     442              10.51
         Electronics  371034.64  40891.38      9436    3061              11.02
            Cameras   267607.69  30289.80       592 

## 9. Product & Profitability Analysis

Product-level analysis is used to identify the products that contribute most to overall sales and to evaluate their profitability.

The analysis includes:

* Product-level sales, quantity, orders, and profit
* ABC classification based on cumulative sales contribution
* Profit margin by product
* Identification of products with the lowest profit margins

ABC classification uses cumulative sales contribution:

* **A:** Products contributing up to 80% of cumulative sales
* **B:** Products contributing from 80% to 95%
* **C:** Products contributing from 95% to 100%

This classification helps identify products that have different levels of importance to overall sales.

In [500]:
product_abc = (
    clean_df
    .groupby(["Product Card Id", "Product Name"])
    .agg(
        Sales=("Sales", "sum"),
        Quantity=("Order Item Quantity", "sum"),
        Orders=("Order Id", "nunique"),
        Profit=("Benefit per order", "sum")
    )
    .reset_index()
)

product_abc = product_abc.sort_values(
    "Sales",
    ascending=False
)

product_abc["Sales %"] = (
    product_abc["Sales"] / product_abc["Sales"].sum() * 100
)

product_abc["Cumulative Sales %"] = (
    product_abc["Sales %"].cumsum()
)

product_abc["ABC Class"] = pd.cut(
    product_abc["Cumulative Sales %"],
    bins=[0, 80, 95, 100],
    labels=["A", "B", "C"],
    include_lowest=True
)

print(product_abc.head(20).round(2).to_string(index=False))

 Product Card Id                                  Product Name      Sales  Quantity  Orders    Profit  Sales %  Cumulative Sales % ABC Class
            1004     Field & Stream Sportsman 16 Gun Fire Safe 6929653.69     17325   15164 756220.77    18.84               18.84         A
             365              Perfect Fitness Perfect Rip Deck 4421143.14     73698   20359 493828.30    12.02               30.86         A
             957 Diamondback Women's Serene Classic Comfort Bi 4118425.57     13729   12299 427455.57    11.20               42.05         A
             191             Nike Men's Free 5.0+ Running Shoe 3667633.20     36680   11092 379915.82     9.97               52.02         A
             502          Nike Men's Dri-FIT Victory Golf Polo 3147800.00     62956   17869 350421.03     8.56               60.58         A
            1073                   Pelican Sunstream 100 Kayak 3099845.09     15500   13727 324076.37     8.43               69.01         A
             

In [501]:
abc_summary = (
    product_abc
    .groupby("ABC Class", observed=True)
    .agg(
        Products=("Product Card Id", "count"),
        Sales=("Sales", "sum"),
        Quantity=("Quantity", "sum"),
        Orders=("Orders", "sum"),
        Profit=("Profit", "sum")
    )
    .reset_index()
)

abc_summary["Sales %"] = (
    abc_summary["Sales"] / product_abc["Sales"].sum() * 100
)

print(abc_summary.round(2).to_string(index=False))

ABC Class  Products       Sales  Quantity  Orders     Profit  Sales %
        A         7 28276258.35    242134  109293 3043820.67    76.87
        B        16  6653168.30     97763   32744  729353.87    18.09
        C        95  1855308.37     44182   17726  193728.43     5.04


In [502]:
product_abc["Profit Margin (%)"] = (
    product_abc["Profit"] / product_abc["Sales"] * 100
)

profitability_analysis = product_abc.sort_values(
    "Profit",
    ascending=False
)

print(
    profitability_analysis[
        [
            "Product Card Id",
            "Product Name",
            "Sales",
            "Profit",
            "Profit Margin (%)",
            "ABC Class"
        ]
    ]
    .head(20)
    .round(2)
    .to_string(index=False)
)

 Product Card Id                                  Product Name      Sales    Profit  Profit Margin (%) ABC Class
            1004     Field & Stream Sportsman 16 Gun Fire Safe 6929653.69 756220.77              10.91         A
             365              Perfect Fitness Perfect Rip Deck 4421143.14 493828.30              11.17         A
             957 Diamondback Women's Serene Classic Comfort Bi 4118425.57 427455.57              10.38         A
             191             Nike Men's Free 5.0+ Running Shoe 3667633.20 379915.82              10.36         A
             502          Nike Men's Dri-FIT Victory Golf Polo 3147800.00 350421.03              11.13         A
            1073                   Pelican Sunstream 100 Kayak 3099845.09 324076.37              10.45         A
            1014              O'Brien Men's Neoprene Life Vest 2888993.91 318451.43              11.02         B
             403       Nike Men's CJ Elite 2 TD Football Cleat 2891757.66 311902.82             

In [503]:
low_margin_products = (
    product_abc
    .sort_values("Profit Margin (%)")
)

print(
    low_margin_products[
        [
            "Product Card Id",
            "Product Name",
            "Sales",
            "Profit",
            "Profit Margin (%)",
            "ABC Class"
        ]
    ]
    .head(20)
    .round(2)
    .to_string(index=False)
)

 Product Card Id                                  Product Name    Sales  Profit  Profit Margin (%) ABC Class
             860        Bushnell Pro X7 Jolt Slope Rangefinder  6599.89 -255.95              -3.88         C
             208                           SOLE E35 Elliptical 29999.85 -965.12              -3.22         C
              60                           SOLE E25 Elliptical  9999.90 -169.56              -1.70         C
             203             GoPro HERO3+ Black Edition Camera 12799.68  245.63               1.92         C
             303             Garmin Forerunner 910XT GPS Watch 13999.65  391.13               2.79         C
              61 Diamondback Girls' Clarity 24 Hybrid Bike 201  8399.72  284.42               3.39         C
             359       Nike Men's Free TR 5.0 TB Training Shoe 20597.94  714.43               3.47         C
             705 Cleveland Golf Women's 588 RTX CB Satin Chrom  8399.30  370.61               4.41         C
            1357   

## 10. Demand Analysis

Demand analysis evaluates how consistently individual products are purchased over time.

Monthly product demand is calculated first, followed by the coefficient of variation (CV) to measure demand variability.

The analysis uses historical data through September 2017 so that the demand measures remain consistent with the subsequent inventory planning analysis.

The coefficient of variation is calculated as:

* **CV < 0.20:** Low demand variability
* **0.20 ≤ CV < 0.50:** Medium demand variability
* **CV ≥ 0.50:** High demand variability

In [504]:
monthly_demand = (
    clean_df
    .groupby("Order Year-Month")
    .agg(
        Sales=("Sales", "sum"),
        Quantity=("Order Item Quantity", "sum"),
        Orders=("Order Id", "nunique")
    )
    .reset_index()
)

print(
    monthly_demand
    .tail(24)
    .round(2)
    .to_string(index=False)
)

Order Year-Month      Sales  Quantity  Orders
         2016-02  968542.85     10765    1650
         2016-03 1025853.12     11349    1764
         2016-04 1001211.58     11208    1706
         2016-05 1029400.20     11603    1763
         2016-06 1003059.55     11008    1690
         2016-07 1045715.62     11652    1758
         2016-08 1048200.25     11683    1766
         2016-09 1002397.04     11284    1730
         2016-10 1048130.55     11936    1776
         2016-11 1047590.14     11493    1721
         2016-12 1037408.17     11774    1763
         2017-01 1029698.02     11605    1745
         2017-02  992534.90     11070    1614
         2017-03 1048004.78     11676    1782
         2017-04 1038321.62     11189    1739
         2017-05 1105485.32     11033    1763
         2017-06 1032086.49     10194    1676
         2017-07 1104373.36     11091    1776
         2017-08 1109337.17     11095    1768
         2017-09 1143775.11     10502    1723
         2017-10 1073994.17      2

In [505]:
monthly_coverage = (
    clean_df
    .groupby("Order Year-Month")
    .agg(
        Order_Items=("Order Item Id", "count"),
        Orders=("Order Id", "nunique"),
        Products=("Product Card Id", "nunique")
    )
    .reset_index()
)

print(
    monthly_coverage
    .tail(24)
    .to_string(index=False)
)

Order Year-Month  Order_Items  Orders  Products
         2016-02         4894    1650        54
         2016-03         5210    1764        54
         2016-04         5097    1706        54
         2016-05         5302    1763        54
         2016-06         5054    1690        54
         2016-07         5305    1758        54
         2016-08         5334    1766        54
         2016-09         5160    1730        54
         2016-10         5398    1776        54
         2016-11         5210    1721        54
         2016-12         5269    1763        54
         2017-01         5217    1745        54
         2017-02         4906    1614        54
         2017-03         5347    1782        54
         2017-04         5212    1739        87
         2017-05         5317    1763        42
         2017-06         4951    1676        42
         2017-07         5318    1776        42
         2017-08         5305    1768        42
         2017-09         5189    1723   

In [506]:
product_monthly = (
    clean_df
    .groupby(["Product Card Id", "Product Name", "Order Year-Month"])
    .agg(
        Quantity=("Order Item Quantity", "sum"),
        Sales=("Sales", "sum"),
        Orders=("Order Id", "nunique")
    )
    .reset_index()
)

product_month_count = (
    product_monthly
    .groupby(["Product Card Id", "Product Name"])
    .agg(
        Active_Months=("Order Year-Month", "nunique")
    )
    .reset_index()
)

print(
    product_month_count
    .sort_values("Active_Months", ascending=False)
    .head(20)
    .to_string(index=False)
)

 Product Card Id                                  Product Name  Active_Months
             191             Nike Men's Free 5.0+ Running Shoe             34
             502          Nike Men's Dri-FIT Victory Golf Polo             34
             627 Under Armour Girls' Toddler Spine Surge Runni             34
             403       Nike Men's CJ Elite 2 TD Football Cleat             34
             365              Perfect Fitness Perfect Rip Deck             34
            1014              O'Brien Men's Neoprene Life Vest             34
             957 Diamondback Women's Serene Classic Comfort Bi             34
            1073                   Pelican Sunstream 100 Kayak             34
            1004     Field & Stream Sportsman 16 Gun Fire Safe             34
             116                    Nike Men's Comfort 2 Slide             28
             249 Under Armour Women's Micro G Skulpt Running S             28
             235   Under Armour Hustle Storm Medium Duffle Bag  

In [507]:
top_products = (
    product_month_count
    .sort_values("Active_Months", ascending=False)
    .head(9)["Product Card Id"]
    .tolist()
)

top_product_demand = (
    product_monthly[
        product_monthly["Product Card Id"].isin(top_products)
    ]
    .sort_values(["Product Card Id", "Order Year-Month"])
)

print(
    top_product_demand[
        ["Product Card Id", "Product Name", "Order Year-Month", "Quantity"]
    ]
    .to_string(index=False)
)

 Product Card Id                                  Product Name Order Year-Month  Quantity
             191             Nike Men's Free 5.0+ Running Shoe          2015-01      1138
             191             Nike Men's Free 5.0+ Running Shoe          2015-02       946
             191             Nike Men's Free 5.0+ Running Shoe          2015-03      1120
             191             Nike Men's Free 5.0+ Running Shoe          2015-04      1021
             191             Nike Men's Free 5.0+ Running Shoe          2015-05      1371
             191             Nike Men's Free 5.0+ Running Shoe          2015-06      1238
             191             Nike Men's Free 5.0+ Running Shoe          2015-07      1073
             191             Nike Men's Free 5.0+ Running Shoe          2015-08      1117
             191             Nike Men's Free 5.0+ Running Shoe          2015-09      1104
             191             Nike Men's Free 5.0+ Running Shoe          2015-10      1235
          

In [508]:
demand_stats = (
    top_product_demand
    .groupby(["Product Card Id", "Product Name"])
    .agg(
        Avg_Monthly_Demand=("Quantity", "mean"),
        Std_Monthly_Demand=("Quantity", "std"),
        Min_Monthly_Demand=("Quantity", "min"),
        Max_Monthly_Demand=("Quantity", "max"),
        Active_Months=("Order Year-Month", "nunique")
    )
    .reset_index()
)

demand_stats["Demand_CV (%)"] = (
    demand_stats["Std_Monthly_Demand"]
    / demand_stats["Avg_Monthly_Demand"]
    * 100
)

print(
    demand_stats
    .sort_values("Avg_Monthly_Demand", ascending=False)
    .round(2)
    .to_string(index=False)
)

 Product Card Id                                  Product Name  Avg_Monthly_Demand  Std_Monthly_Demand  Min_Monthly_Demand  Max_Monthly_Demand  Active_Months  Demand_CV (%)
             365              Perfect Fitness Perfect Rip Deck             2167.59              378.74                 120                2432             34          17.47
             502          Nike Men's Dri-FIT Victory Golf Polo             1851.65              329.72                  78                2163             34          17.81
            1014              O'Brien Men's Neoprene Life Vest             1700.09              306.66                  68                1995             34          18.04
             191             Nike Men's Free 5.0+ Running Shoe             1078.82              200.05                  64                1371             34          18.54
             627 Under Armour Girls' Toddler Spine Surge Runni              933.38              172.35                  25             

## 11. Product 365 Forecasting

Monthly demand for **Product Card Id 365** is forecast using:

* 3-Month Moving Average
* Simple Exponential Smoothing (SES)

Data through **September 2017** is used, with the final 6 months reserved for testing.

Forecast accuracy is evaluated using **MAE, RMSE, and MAPE**. SES is tested with different α values, and the final model uses **α = 0.5** to generate a 6-month forecast.

>Note: It is the case study on highest-volume product.

### 11.1 Prepare Forecast Data

In [509]:
product_365 = (
    top_product_demand[
        top_product_demand["Product Card Id"] == 365
    ][
        ["Order Year-Month", "Quantity"]
    ]
    .copy()
)

product_365["Order Year-Month"] = pd.to_datetime(
    product_365["Order Year-Month"]
)

product_365 = product_365.sort_values("Order Year-Month")

print(product_365.to_string(index=False))

Order Year-Month  Quantity
      2015-01-01      2390
      2015-02-01      2005
      2015-03-01      2355
      2015-04-01      2187
      2015-05-01      2220
      2015-06-01      2142
      2015-07-01      2274
      2015-08-01      2411
      2015-09-01      2177
      2015-10-01      2171
      2015-11-01      2205
      2015-12-01      2218
      2016-01-01      2188
      2016-02-01      2047
      2016-03-01      2401
      2016-04-01      2225
      2016-05-01      2432
      2016-06-01      2181
      2016-07-01      2228
      2016-08-01      2311
      2016-09-01      2224
      2016-10-01      2278
      2016-11-01      2260
      2016-12-01      2361
      2017-01-01      2270
      2017-02-01      2136
      2017-03-01      2215
      2017-04-01      2224
      2017-05-01      2218
      2017-06-01      1934
      2017-07-01      2305
      2017-08-01      2313
      2017-09-01      2072
      2017-10-01       120


In [510]:
product_365["3M_Moving_Average"] = (
    product_365["Quantity"]
    .rolling(window=3)
    .mean()
)

print(
    product_365[
        ["Order Year-Month", "Quantity", "3M_Moving_Average"]
    ]
    .tail(12)
    .round(2)
    .to_string(index=False)
)

Order Year-Month  Quantity  3M_Moving_Average
      2016-11-01      2260            2254.00
      2016-12-01      2361            2299.67
      2017-01-01      2270            2297.00
      2017-02-01      2136            2255.67
      2017-03-01      2215            2207.00
      2017-04-01      2224            2191.67
      2017-05-01      2218            2219.00
      2017-06-01      1934            2125.33
      2017-07-01      2305            2152.33
      2017-08-01      2313            2184.00
      2017-09-01      2072            2230.00
      2017-10-01       120            1501.67


C:\Users\ok\AppData\Local\Temp\ipykernel_13264\1708231443.py:12: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


In [511]:
print(
    product_365[
        ["Order Year-Month", "Quantity"]
    ].to_string(index=False)
)

Order Year-Month  Quantity
      2015-01-01      2390
      2015-02-01      2005
      2015-03-01      2355
      2015-04-01      2187
      2015-05-01      2220
      2015-06-01      2142
      2015-07-01      2274
      2015-08-01      2411
      2015-09-01      2177
      2015-10-01      2171
      2015-11-01      2205
      2015-12-01      2218
      2016-01-01      2188
      2016-02-01      2047
      2016-03-01      2401
      2016-04-01      2225
      2016-05-01      2432
      2016-06-01      2181
      2016-07-01      2228
      2016-08-01      2311
      2016-09-01      2224
      2016-10-01      2278
      2016-11-01      2260
      2016-12-01      2361
      2017-01-01      2270
      2017-02-01      2136
      2017-03-01      2215
      2017-04-01      2224
      2017-05-01      2218
      2017-06-01      1934
      2017-07-01      2305
      2017-08-01      2313
      2017-09-01      2072
      2017-10-01       120


### 11.2 Train-Test Split

27 months are used for training and the final 6 months for testing.

In [512]:
forecast_data = product_365[
    product_365["Order Year-Month"] <= "2017-09-01"
].copy()

forecast_data = forecast_data[
    ["Order Year-Month", "Quantity"]
].reset_index(drop=True)

print(forecast_data.tail(10).to_string(index=False))

print("\nNumber of months:", len(forecast_data))

Order Year-Month  Quantity
      2016-12-01      2361
      2017-01-01      2270
      2017-02-01      2136
      2017-03-01      2215
      2017-04-01      2224
      2017-05-01      2218
      2017-06-01      1934
      2017-07-01      2305
      2017-08-01      2313
      2017-09-01      2072

Number of months: 33


In [513]:
train = forecast_data.iloc[:-6].copy()
test = forecast_data.iloc[-6:].copy()

print("Training period:")
print(train["Order Year-Month"].min(), "to", train["Order Year-Month"].max())

print("\nTest period:")
print(test["Order Year-Month"].min(), "to", test["Order Year-Month"].max())

print("\nTraining months:", len(train))
print("Test months:", len(test))

Training period:
2015-01-01 00:00:00 to 2017-03-01 00:00:00

Test period:
2017-04-01 00:00:00 to 2017-09-01 00:00:00

Training months: 27
Test months: 6


### 11.3 Moving Average Forecast

A 3-month moving average is used as the baseline forecast.

In [514]:
# Combine training and test demand so we can calculate
# a rolling 3-month forecast
combined = pd.concat([
    train[["Order Year-Month", "Quantity"]],
    test[["Order Year-Month", "Quantity"]]
]).reset_index(drop=True)

# Calculate the 3-month moving average
combined["Forecast"] = (
    combined["Quantity"]
    .rolling(window=3)
    .mean()
    .shift(1)
)

# Get forecasts corresponding only to the test period
test = combined.iloc[len(train):].copy()

print(
    test[
        ["Order Year-Month", "Quantity", "Forecast"]
    ]
    .round(2)
    .to_string(index=False)
)

Order Year-Month  Quantity  Forecast
      2017-04-01      2224   2207.00
      2017-05-01      2218   2191.67
      2017-06-01      1934   2219.00
      2017-07-01      2305   2125.33
      2017-08-01      2313   2152.33
      2017-09-01      2072   2184.00


C:\Users\ok\AppData\Local\Temp\ipykernel_13264\2838626683.py:23: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


In [515]:
mae = (
    test["Quantity"] - test["Forecast"]
).abs().mean()

print("MAE:", round(mae, 2))
rmse = np.sqrt(
    ((test["Quantity"] - test["Forecast"]) ** 2).mean()
)
ma3=[round(mae, 2),round(rmse, 2)]
print("RMSE:", round(rmse, 2))

MAE: 130.11
RMSE: 159.61


In [516]:
test["Error"] = test["Quantity"] - test["Forecast"]
test["Absolute Error"] = test["Error"].abs()
test["Percentage Error (%)"] = (
    test["Absolute Error"] / test["Quantity"] * 100
)

print(
    test[
        [
            "Order Year-Month",
            "Quantity",
            "Forecast",
            "Error",
            "Percentage Error (%)"
        ]
    ].round(2).to_string(index=False)
)

Order Year-Month  Quantity  Forecast   Error  Percentage Error (%)
      2017-04-01      2224   2207.00   17.00                  0.76
      2017-05-01      2218   2191.67   26.33                  1.19
      2017-06-01      1934   2219.00 -285.00                 14.74
      2017-07-01      2305   2125.33  179.67                  7.79
      2017-08-01      2313   2152.33  160.67                  6.95
      2017-09-01      2072   2184.00 -112.00                  5.41


C:\Users\ok\AppData\Local\Temp\ipykernel_13264\2527492823.py:16: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(2).to_string(index=False)


In [517]:
mape = test["Percentage Error (%)"].mean()
ma3.append(round(mape, 2))

print("MAPE:", round(mape, 2), "%")

MAPE: 6.14 %


### 11.4 SES Forecast

Simple Exponential Smoothing is evaluated across α values from **0.1 to 0.9**.

In [518]:
alpha = 0.3

forecast = [train["Quantity"].iloc[0]]

for i in range(1, len(train)):
    next_forecast = (
        alpha * train["Quantity"].iloc[i - 1]
        + (1 - alpha) * forecast[i - 1]
    )
    
    forecast.append(next_forecast)

train["SES Forecast"] = forecast

test_forecast = forecast[-1]

test["SES Forecast"] = test_forecast

print(
    test[
        ["Order Year-Month", "Quantity", "SES Forecast"]
    ].round(2).to_string(index=False)
)

Order Year-Month  Quantity  SES Forecast
      2017-04-01      2224       2240.14
      2017-05-01      2218       2240.14
      2017-06-01      1934       2240.14
      2017-07-01      2305       2240.14
      2017-08-01      2313       2240.14
      2017-09-01      2072       2240.14


C:\Users\ok\AppData\Local\Temp\ipykernel_13264\1995069124.py:22: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(2).to_string(index=False)


In [519]:
mae_ses = (
    test["Quantity"] - test["SES Forecast"]
).abs().mean()

rmse_ses = np.sqrt(
    ((test["Quantity"] - test["SES Forecast"]) ** 2).mean()
)

mape_ses = (
    (test["Quantity"] - test["SES Forecast"]).abs()
    / test["Quantity"]
).mean() * 100

print("SES MAE:", round(mae_ses, 2))
print("SES RMSE:", round(rmse_ses, 2))
print("SES MAPE:", round(mape_ses, 2), "%")

SES MAE: 108.38
SES RMSE: 148.47
SES MAPE: 5.27 %


In [520]:
alpha_values = np.arange(0.1, 1.0, 0.1)

results = []

for alpha in alpha_values:

    forecast = [train["Quantity"].iloc[0]]

    for i in range(1, len(train)):
        next_forecast = (
            alpha * train["Quantity"].iloc[i - 1]
            + (1 - alpha) * forecast[i - 1]
        )

        forecast.append(next_forecast)

    ses_forecast = forecast[-1]

    errors = test["Quantity"] - ses_forecast

    mae = errors.abs().mean()

    rmse = np.sqrt(
        (errors ** 2).mean()
    )

    mape = (
        errors.abs() / test["Quantity"]
    ).mean() * 100

    results.append([
        alpha,
        mae,
        rmse,
        mape
    ])

alpha_results = pd.DataFrame(
    results,
    columns=[
        "Alpha",
        "MAE",
        "RMSE",
        "MAPE"
    ]
)

print(
    alpha_results.round(2).to_string(index=False)
)

 Alpha    MAE   RMSE  MAPE
   0.1 114.47 157.04  5.59
   0.2 111.32 152.39  5.42
   0.3 108.38 148.47  5.27
   0.4 104.26 143.70  5.06
   0.5 104.53 139.34  5.04
   0.6 109.72 136.19  5.24
   0.7 115.09 134.75  5.44
   0.8 120.45 135.22  5.65
   0.9 125.60 137.46  5.85


In [521]:
forecast_model_comparison = pd.DataFrame({
    "Model": [
        "3-Month Moving Average",
        "SES (α=0.3)",
        "SES (α=0.5)"
    ],
    "MAE": [
        ma3[0],
        alpha_results.loc[2, "MAE"],
        alpha_results.loc[4, "MAE"]
    ],
    "RMSE": [
        ma3[1],
        alpha_results.loc[2, "RMSE"],
        alpha_results.loc[4, "RMSE"]
    ],
    "MAPE": [
        ma3[2],
        alpha_results.loc[2, "MAPE"],
        alpha_results.loc[4, "MAPE"]
    ]
})

forecast_model_comparison

,Model,MAE,RMSE,MAPE
0,3-Month Moving Average,130.110000,159.610000,6.140000
1,SES (α=0.3),108.378349,148.467237,5.271924
2,SES (α=0.5),104.534613,139.344246,5.039240


### 11.5 Final Forecast

The final SES model with **α = 0.5** is used to forecast demand for the next 6 months.

In [522]:
alpha = 0.5

final_forecast = [forecast_data["Quantity"].iloc[0]]

for i in range(1, len(forecast_data)):
    next_forecast = (
        alpha * forecast_data["Quantity"].iloc[i - 1]
        + (1 - alpha) * final_forecast[i - 1]
    )

    final_forecast.append(next_forecast)

forecast_data["SES Forecast"] = final_forecast

last_forecast = forecast_data["SES Forecast"].iloc[-1]

print(
    "Final SES forecast:",
    round(last_forecast, 2))

Final SES forecast: 2251.82


In [523]:
last_date = forecast_data["Order Year-Month"].iloc[-1]

future_dates = pd.date_range(
    start=last_date + pd.DateOffset(months=1),
    periods=6,
    freq="MS"
)

future_forecast = pd.DataFrame({
    "Order Year-Month": future_dates,
    "Forecast Quantity": round(last_forecast)
})

print(
    future_forecast.to_string(index=False)
)

Order Year-Month  Forecast Quantity
      2017-10-01               2252
      2017-11-01               2252
      2017-12-01               2252
      2018-01-01               2252
      2018-02-01               2252
      2018-03-01               2252


>Note: It is a known limitation of the method chosen that the forecast remains flat for all 6 months. 

### 11.6 Forecast Output Preparation

The Product 365 forecast is prepared as a separate reporting table containing historical demand and future forecast values.

This table can be used directly in Power BI to visualize historical demand alongside the forecast.

In [524]:
forecast_df = pd.DataFrame({
    "Month": pd.date_range(
        start="2017-10-01",
        periods=6,
        freq="MS"
    ),
    "Forecast Demand": [round(last_forecast, 2)] * 6
})

forecast_df

,Month,Forecast Demand
0,2017-10-01,2251.82
1,2017-11-01,2251.82
2,2017-12-01,2251.82
3,2018-01-01,2251.82
4,2018-02-01,2251.82
5,2018-03-01,2251.82


## 12. Inventory Planning

Inventory planning combines historical demand, lead-time estimates, safety stock, reorder points, EOQ, and ABC-demand variability classification.

Product 365 is used as the detailed example, while the broader product-level analysis identifies inventory priorities.

### 12.1 Product 365 Daily Demand

Daily demand is calculated through September 2017, including zero-demand days.

In [525]:
product_365_daily = (
    clean_df[
        clean_df["Product Card Id"] == 365
    ]
    .groupby("Order Date")["Order Item Quantity"]
    .sum()
    .reset_index()
)

product_365_daily["Order Date"] = pd.to_datetime(
    product_365_daily["Order Date"]
)

print(
    product_365_daily["Order Item Quantity"]
    .describe()
    .round(2)
    .to_string()
)

count    1006.00
mean       73.26
std        16.72
min        31.00
25%        62.00
50%        72.00
75%        84.00
max       133.00


### 12.2 Safety Stock & Reorder Point

Using a 95% service-level assumption:

* Average Daily Demand
* Average Lead Time
* Lead-Time Demand
* Safety Stock
* Reorder Point

In [526]:
lead_time_stats = clean_df["Days for shipping (real)"].describe()

print(
    lead_time_stats.round(2).to_string()
)

count    180519.00
mean          3.50
std           1.62
min           0.00
25%           2.00
50%           3.00
75%           5.00
max           6.00


In [527]:
z = 1.645

daily_demand_std = product_365_daily[
    "Order Item Quantity"
].std()

average_lead_time = clean_df[
    "Days for shipping (real)"
].mean()

safety_stock = (
    z
    * daily_demand_std
    * np.sqrt(average_lead_time)
)

print("Daily Demand Std:", round(daily_demand_std, 2))
print("Average Lead Time:", round(average_lead_time, 2), "days")
print("Safety Stock:", round(safety_stock, 2), "units")

Daily Demand Std: 16.72
Average Lead Time: 3.5 days
Safety Stock: 51.43 units


In [528]:
average_daily_demand = product_365_daily[
    "Order Item Quantity"
].mean()

reorder_point = (
    average_daily_demand * average_lead_time
    + safety_stock)

print(
    "Average Daily Demand:",
    round(average_daily_demand, 2))

print(
    "Lead-Time Demand:",
    round(
        average_daily_demand * average_lead_time,
        2))

print(
    "Safety Stock:",
    round(safety_stock, 2))

print(
    "Reorder Point:",
    round(reorder_point, 2),
    "units")

Average Daily Demand: 73.26
Lead-Time Demand: 256.23
Safety Stock: 51.43
Reorder Point: 307.66 units


In [529]:
inventory_planning = pd.DataFrame({
    "Metric": [
        "Average Daily Demand",
        "Average Lead Time (Days)",
        "Lead-Time Demand",
        "Safety Stock",
        "Reorder Point"
    ],
    "Value": [
        round(average_daily_demand, 2),
        round(average_lead_time, 2),
        round(
                average_daily_demand * average_lead_time,
                2),
        round(safety_stock, 0),
        round(reorder_point, 0)
    ]
})

inventory_planning

,Metric,Value
0,Average Daily Demand,73.26
1,Average Lead Time (Days),3.50
2,Lead-Time Demand,256.23
3,Safety Stock,51.00
4,Reorder Point,308.00


### 12.3 Economic Order Quantity

EOQ is calculated using annualized demand and assumed ordering and holding costs.

In [530]:
product_365_monthly = (
    clean_df[
        (clean_df["Product Card Id"] == 365) &
        (clean_df["Order Year-Month"] <= "2017-09")
    ]
    .groupby("Order Year-Month")["Order Item Quantity"]
    .sum()
    .reset_index()
)

average_monthly_demand = (
    product_365_monthly["Order Item Quantity"].mean()
)

annual_demand = average_monthly_demand * 12

print("Average Monthly Demand:",
      round(average_monthly_demand, 2))

print("Estimated Annual Demand:",
      round(annual_demand, 2), "units")

Average Monthly Demand: 2229.64
Estimated Annual Demand: 26755.64 units


>Note: Ordering and Holding costs are assumed since we do not have actual costs.

In [531]:
D = annual_demand
S = 500
H = 50

eoq = np.sqrt((2 * D * S) / H)

print("Annual Demand:", round(D, 2), "units")
print("Ordering Cost:", S)
print("Holding Cost:", H)
print("EOQ:", round(eoq, 2), "units")

Annual Demand: 26755.64 units
Ordering Cost: 500
Holding Cost: 50
EOQ: 731.51 units


In [532]:
orders_per_year = annual_demand / eoq

days_per_order = 365 / orders_per_year

print("Orders per Year:", round(orders_per_year, 2))
print("Days Between Orders:", round(days_per_order, 2))

Orders per Year: 36.58
Days Between Orders: 9.98


In [533]:
product_monthly_demand = (
    clean_df[
        clean_df["Order Year-Month"] <= "2017-09"
    ]
    .groupby(
        ["Product Card Id", "Product Name", "Order Year-Month"]
    )["Order Item Quantity"]
    .sum()
    .reset_index()
)

product_cv = (
    product_monthly_demand
    .groupby(["Product Card Id", "Product Name"])
    .agg(
        Average_Monthly_Demand=("Order Item Quantity", "mean"),
        Demand_Std=("Order Item Quantity", "std")
    )
    .reset_index()
)

product_cv["CV"] = (
    product_cv["Demand_Std"]
    / product_cv["Average_Monthly_Demand"]
)

print(product_cv.sort_values("CV", ascending=False).head(10).round(2).to_string(index=False))

 Product Card Id                                  Product Name  Average_Monthly_Demand  Demand_Std   CV
             647 Merrell Women's Siren Mid Waterproof Hiking B                   11.67        8.33 0.71
             715              TaylorMade Women's RBZ SL Rescue                   11.33        7.99 0.71
             625  Nike Men's Kobe IX Elite Low Basketball Shoe                   10.50        7.26 0.69
             306                  Polar FT4 Heart Rate Monitor                   32.67       22.10 0.68
             777                        Bag Boy M330 Push Cart                   34.67       23.24 0.67
             216 Yakima DoubleDown Ace Hitch Mount 4-Bike Rack                   10.67        7.03 0.66
             691             MDGolf Pittsburgh Penguins Putter                   29.00       18.12 0.62
             607             Garmin Approach S3 Golf GPS Watch                   12.83        7.99 0.62
              24                   Elevation Training Mask 2.0  

In [534]:
actual_df = product_365_monthly[
    product_365_monthly["Order Year-Month"] <= "2017-09"
].copy()
actual_df = actual_df.rename(
    columns={
        "Order Year-Month": "Month",
        "Order Item Quantity": "Actual Demand"
    }
)
actual_df["Month"] = pd.to_datetime(
    actual_df["Month"]
)
forecast_df["Actual Demand"] = None

actual_df["Forecast Demand"] = None

actual_forecast_df = pd.concat(
    [
        actual_df[["Month", "Actual Demand", "Forecast Demand"]],
        forecast_df[["Month", "Actual Demand", "Forecast Demand"]]
    ],
    ignore_index=True
)

actual_forecast_df = actual_forecast_df.sort_values("Month")

actual_forecast_df

,Month,Actual Demand,Forecast Demand
0,2015-01-01,2390,None
1,2015-02-01,2005,None
2,2015-03-01,2355,None
3,2015-04-01,2187,None
4,2015-05-01,2220,None
5,2015-06-01,2142,None
6,2015-07-01,2274,None
7,2015-08-01,2411,None
8,2015-09-01,2177,None
9,2015-10-01,2171,None


### 12.4 Product-Level Inventory Metrics

Inventory metrics are calculated for each product using demand variability and lead-time estimates.


In [535]:
inventory_analysis = product_abc[
    [
        "Product Card Id",
        "Product Name",
        "ABC Class",
        "Sales",
        "Quantity",
        "Profit",
        "Profit Margin (%)"
    ]
].merge(
    product_cv[
        [
            "Product Card Id",
            "Average_Monthly_Demand",
            "Demand_Std",
            "CV"
        ]
    ],
    on="Product Card Id",
    how="left"
)

inventory_analysis = inventory_analysis.sort_values(
    ["ABC Class", "CV"],
    ascending=[True, False]
)

print(
    inventory_analysis.head(20).round(2).to_string(index=False)
)

 Product Card Id                                  Product Name ABC Class      Sales  Quantity    Profit  Profit Margin (%)  Average_Monthly_Demand  Demand_Std   CV
             191             Nike Men's Free 5.0+ Running Shoe         A 3667633.20     36680 379915.82              10.36                 1109.58       90.07 0.08
            1073                   Pelican Sunstream 100 Kayak         A 3099845.09     15500 324076.37              10.45                  468.94       26.87 0.06
             502          Nike Men's Dri-FIT Victory Golf Polo         A 3147800.00     62956 350421.03              11.13                 1905.39      104.04 0.05
             365              Perfect Fitness Perfect Rip Deck         A 4421143.14     73698 493828.30              11.17                 2229.64      113.73 0.05
             403       Nike Men's CJ Elite 2 TD Football Cleat         A 2891757.66     22246 311902.82              10.79                  673.15       33.70 0.05
            1004

In [536]:
inventory_analysis["Demand Variability"] = np.select(
    [
        inventory_analysis["CV"] < 0.20,
        inventory_analysis["CV"] < 0.50,
        inventory_analysis["CV"] >= 0.50
    ],
    [
        "Low",
        "Medium",
        "High"
    ],
    default="Unknown"
)

print(
    inventory_analysis[
        [
            "Product Card Id",
            "Product Name",
            "ABC Class",
            "Average_Monthly_Demand",
            "CV",
            "Demand Variability"
        ]
    ]
    .sort_values(["ABC Class", "CV"], ascending=[True, False])
    .head(30)
    .round(2)
    .to_string(index=False)
)

 Product Card Id                                  Product Name ABC Class  Average_Monthly_Demand   CV Demand Variability
             191             Nike Men's Free 5.0+ Running Shoe         A                 1109.58 0.08                Low
            1073                   Pelican Sunstream 100 Kayak         A                  468.94 0.06                Low
             502          Nike Men's Dri-FIT Victory Golf Polo         A                 1905.39 0.05                Low
             365              Perfect Fitness Perfect Rip Deck         A                 2229.64 0.05                Low
             403       Nike Men's CJ Elite 2 TD Football Cleat         A                  673.15 0.05                Low
            1004     Field & Stream Sportsman 16 Gun Fire Safe         A                  524.48 0.05                Low
             957 Diamondback Women's Serene Classic Comfort Bi         A                  415.58 0.04                Low
              44    adidas Men's

In [537]:
abc_variability_summary = (
    inventory_analysis[
        inventory_analysis["Demand Variability"] != "Unknown"
    ]
    .groupby(
        ["ABC Class", "Demand Variability"],
        observed=True
    )
    .agg(
        Products=("Product Card Id", "count"),
        Sales=("Sales", "sum"),
        Average_Monthly_Demand=("Average_Monthly_Demand", "sum")
    )
    .reset_index()
)

abc_variability_summary["Sales Share (%)"] = (
    abc_variability_summary["Sales"]
    / inventory_analysis["Sales"].sum()
    * 100
)

print(
    abc_variability_summary
    .sort_values(
        ["ABC Class", "Demand Variability"]
    )
    .round(2)
    .to_string(index=False)
)

ABC Class Demand Variability  Products       Sales  Average_Monthly_Demand  Sales Share (%)
        A                Low         7 28276258.35                 7326.76            76.87
        B                Low         2  4158076.58                 2710.45            11.30
        B             Medium         3   185650.61                  101.93             0.50
        C               High        24   331555.26                  472.00             0.90
        C             Medium        51  1219873.22                 1522.27             3.32


In [538]:
product_daily_demand = (
    clean_df[
        clean_df["Order Date"] <= pd.to_datetime("2017-09-30").date()
    ]
    .groupby(
        ["Product Card Id", "Product Name", "Order Date"]
    )["Order Item Quantity"]
    .sum()
    .reset_index()
)

print(product_daily_demand.head())
print("\nRows:", len(product_daily_demand))

   Product Card Id                             Product Name  Order Date  \
0               19  Nike Men's Fingertrap Max Training Shoe  2017-04-24   
1               19  Nike Men's Fingertrap Max Training Shoe  2017-04-26   
2               19  Nike Men's Fingertrap Max Training Shoe  2017-05-03   
3               19  Nike Men's Fingertrap Max Training Shoe  2017-05-04   
4               19  Nike Men's Fingertrap Max Training Shoe  2017-05-06   

   Order Item Quantity  
0                    1  
1                    1  
2                    1  
3                    1  
4                    1  

Rows: 22088


In [539]:
product_daily_demand["Order Date"] = pd.to_datetime(
    product_daily_demand["Order Date"]
)

all_dates = pd.date_range(
    start=product_daily_demand["Order Date"].min(),
    end=product_daily_demand["Order Date"].max(),
    freq="D"
)

# Get each product's first recorded order date
product_start_dates = (
    product_daily_demand
    .groupby(
        ["Product Card Id", "Product Name"]
    )["Order Date"]
    .min()
    .reset_index()
)

# Create a product-specific date range
product_dates = []

for _, row in product_start_dates.iterrows():
    dates = pd.date_range(
        start=row["Order Date"],
        end="2017-09-30",
        freq="D"
    )

    temp = pd.DataFrame({
        "Product Card Id": row["Product Card Id"],
        "Product Name": row["Product Name"],
        "Order Date": dates
    })

    product_dates.append(temp)

product_dates = pd.concat(
    product_dates,
    ignore_index=True
)

# Add demand and fill days with no sales as zero
product_daily_demand = product_dates.merge(
    product_daily_demand,
    on=["Product Card Id", "Product Name", "Order Date"],
    how="left"
)

product_daily_demand["Order Item Quantity"] = (
    product_daily_demand["Order Item Quantity"]
    .fillna(0)
)

print(product_daily_demand.head(10))

print("\nRows:", len(product_daily_demand))

print(
    "Zero-demand days:",
    (product_daily_demand["Order Item Quantity"] == 0).sum()
)

   Product Card Id                             Product Name Order Date  \
0               19  Nike Men's Fingertrap Max Training Shoe 2017-04-24   
1               19  Nike Men's Fingertrap Max Training Shoe 2017-04-25   
2               19  Nike Men's Fingertrap Max Training Shoe 2017-04-26   
3               19  Nike Men's Fingertrap Max Training Shoe 2017-04-27   
4               19  Nike Men's Fingertrap Max Training Shoe 2017-04-28   
5               19  Nike Men's Fingertrap Max Training Shoe 2017-04-29   
6               19  Nike Men's Fingertrap Max Training Shoe 2017-04-30   
7               19  Nike Men's Fingertrap Max Training Shoe 2017-05-01   
8               19  Nike Men's Fingertrap Max Training Shoe 2017-05-02   
9               19  Nike Men's Fingertrap Max Training Shoe 2017-05-03   

   Order Item Quantity  
0                  1.0  
1                  0.0  
2                  1.0  
3                  0.0  
4                  0.0  
5                  0.0  
6         

In [540]:
daily_demand_stats = (
    product_daily_demand
    .groupby(
        ["Product Card Id", "Product Name"]
    )["Order Item Quantity"]
    .agg(
        Average_Daily_Demand="mean",
        Daily_Demand_Std="std",
        Minimum_Daily_Demand="min",
        Maximum_Daily_Demand="max"
    )
    .reset_index()
)

print(
    daily_demand_stats
    .sort_values("Average_Daily_Demand", ascending=False)
    .head(20)
    .round(2)
    .to_string(index=False)
)

 Product Card Id                                  Product Name  Average_Daily_Demand  Daily_Demand_Std  Minimum_Daily_Demand  Maximum_Daily_Demand
             365              Perfect Fitness Perfect Rip Deck                 73.28             16.68                  31.0                 133.0
             502          Nike Men's Dri-FIT Victory Golf Polo                 62.63             15.95                  15.0                 119.0
            1014              O'Brien Men's Neoprene Life Vest                 57.50             14.78                  16.0                 116.0
             191             Nike Men's Free 5.0+ Running Shoe                 36.47             12.09                   5.0                  82.0
             627 Under Armour Girls' Toddler Spine Surge Runni                 31.58             10.65                   8.0                  68.0
             403       Nike Men's CJ Elite 2 TD Football Cleat                 22.13              4.85                

In [541]:
z = 1.645

average_lead_time = clean_df["Days for shipping (real)"].mean()

inventory_metrics = daily_demand_stats.copy()

inventory_metrics["Lead_Time_Demand"] = (
    inventory_metrics["Average_Daily_Demand"]
    * average_lead_time
)

inventory_metrics["Safety_Stock"] = (
    z
    * inventory_metrics["Daily_Demand_Std"]
    * np.sqrt(average_lead_time)
)

inventory_metrics["Reorder_Point"] = (
    inventory_metrics["Lead_Time_Demand"]
    + inventory_metrics["Safety_Stock"]
)

print(
    inventory_metrics[
        [
            "Product Card Id",
            "Product Name",
            "Average_Daily_Demand",
            "Daily_Demand_Std",
            "Lead_Time_Demand",
            "Safety_Stock",
            "Reorder_Point"
        ]
    ]
    .sort_values("Reorder_Point", ascending=False)
    .head(20)
    .round(2)
    .to_string(index=False)
)

 Product Card Id                                  Product Name  Average_Daily_Demand  Daily_Demand_Std  Lead_Time_Demand  Safety_Stock  Reorder_Point
             365              Perfect Fitness Perfect Rip Deck                 73.28             16.68            256.33         51.32         307.65
             502          Nike Men's Dri-FIT Victory Golf Polo                 62.63             15.95            219.05         49.06         268.11
            1014              O'Brien Men's Neoprene Life Vest                 57.50             14.78            201.13         45.48         246.61
             191             Nike Men's Free 5.0+ Running Shoe                 36.47             12.09            127.56         37.18         164.74
             627 Under Armour Girls' Toddler Spine Surge Runni                 31.58             10.65            110.47         32.77         143.23
             403       Nike Men's CJ Elite 2 TD Football Cleat                 22.13              4.

### 12.5 Inventory Priority

Products are classified using their **ABC Class** and **Demand Variability** to identify inventory priorities.

In [542]:
inventory_analysis = inventory_analysis.merge(
    inventory_metrics[
        [
            "Product Card Id",
            "Average_Daily_Demand",
            "Daily_Demand_Std",
            "Lead_Time_Demand",
            "Safety_Stock",
            "Reorder_Point"
        ]
    ],
    on="Product Card Id",
    how="left"
)

inventory_analysis["Reorder_Point"] = (
    np.ceil(inventory_analysis["Reorder_Point"])
)

inventory_analysis["Safety_Stock"] = (
    np.ceil(inventory_analysis["Safety_Stock"])
)

inventory_analysis = inventory_analysis.sort_values(
    ["ABC Class", "Reorder_Point"],
    ascending=[True, False]
)

print(
    inventory_analysis[
        [
            "Product Card Id",
            "Product Name",
            "ABC Class",
            "Demand Variability",
            "Sales",
            "Average_Monthly_Demand",
            "Average_Daily_Demand",
            "Safety_Stock",
            "Reorder_Point"
        ]
    ]
    .head(30)
    .round(2)
    .to_string(index=False)
)

 Product Card Id                                  Product Name ABC Class Demand Variability      Sales  Average_Monthly_Demand  Average_Daily_Demand  Safety_Stock  Reorder_Point
             365              Perfect Fitness Perfect Rip Deck         A                Low 4421143.14                 2229.64                 73.28          52.0          308.0
             502          Nike Men's Dri-FIT Victory Golf Polo         A                Low 3147800.00                 1905.39                 62.63          50.0          269.0
             191             Nike Men's Free 5.0+ Running Shoe         A                Low 3667633.20                 1109.58                 36.47          38.0          165.0
             403       Nike Men's CJ Elite 2 TD Football Cleat         A                Low 2891757.66                  673.15                 22.13          15.0           93.0
            1004     Field & Stream Sportsman 16 Gun Fire Safe         A                Low 6929653.69        

In [543]:
inventory_analysis["Inventory Priority"] = np.select(
    [
        (inventory_analysis["ABC Class"] == "A") &
        (inventory_analysis["Demand Variability"] == "High"),

        (inventory_analysis["ABC Class"] == "A") &
        (inventory_analysis["Demand Variability"].isin(["Medium", "Low"])),

        (inventory_analysis["ABC Class"] == "B") &
        (inventory_analysis["Demand Variability"] == "High"),

        (inventory_analysis["ABC Class"] == "B") &
        (inventory_analysis["Demand Variability"] == "Medium"),

        (inventory_analysis["ABC Class"] == "B") &
        (inventory_analysis["Demand Variability"] == "Low"),

        (inventory_analysis["ABC Class"] == "C") &
        (inventory_analysis["Demand Variability"] == "High"),

        (inventory_analysis["ABC Class"] == "C") &
        (inventory_analysis["Demand Variability"].isin(["Medium", "Low"]))
    ],
    [
        "Critical",
        "High",
        "High",
        "Medium",
        "Medium",
        "Medium",
        "Low"
    ],
    default="Unknown"
)

priority_summary = (
    inventory_analysis
    .groupby("Inventory Priority", dropna=False)
    .agg(
        Products=("Product Card Id", "count"),
        Sales=("Sales", "sum")
    )
    .reset_index()
)

priority_summary["Sales Share (%)"] = (
    priority_summary["Sales"]
    / inventory_analysis["Sales"].sum()
    * 100
)

print(
    priority_summary
    .sort_values("Sales", ascending=False)
    .round(2)
    .to_string(index=False)
)

Inventory Priority  Products       Sales  Sales Share (%)
              High         7 28276258.35            76.87
            Medium        29  4675282.46            12.71
           Unknown        31  2613320.98             7.10
               Low        51  1219873.22             3.32


In [544]:
inventory_analysis["Safety_Stock_Days"] = (
    inventory_analysis["Safety_Stock"]
    / inventory_analysis["Average_Daily_Demand"]
)

top_inventory_products = (
    inventory_analysis[
        inventory_analysis["Reorder_Point"].notna()
    ]
    .sort_values(
        ["ABC Class", "Sales"],
        ascending=[True, False]
    )
)

print(
    top_inventory_products[
        [
            "Product Card Id",
            "Product Name",
            "ABC Class",
            "Demand Variability",
            "Sales",
            "Average_Daily_Demand",
            "Safety_Stock",
            "Safety_Stock_Days",
            "Reorder_Point"
        ]
    ]
    .head(20)
    .round(2)
    .to_string(index=False)
)

 Product Card Id                                  Product Name ABC Class Demand Variability      Sales  Average_Daily_Demand  Safety_Stock  Safety_Stock_Days  Reorder_Point
            1004     Field & Stream Sportsman 16 Gun Fire Safe         A                Low 6929653.69                 17.24          14.0               0.81           74.0
             365              Perfect Fitness Perfect Rip Deck         A                Low 4421143.14                 73.28          52.0               0.71          308.0
             957 Diamondback Women's Serene Classic Comfort Bi         A                Low 4118425.57                 13.66          12.0               0.88           60.0
             191             Nike Men's Free 5.0+ Running Shoe         A                Low 3667633.20                 36.47          38.0               1.04          165.0
             502          Nike Men's Dri-FIT Victory Golf Polo         A                Low 3147800.00                 62.63          5

## 13. SQL & Power BI Data Preparation

The cleaned dataset is separated into relational tables for SQL analysis and Power BI modeling.

The structure follows:

**Customers → Orders → Order Items ← Products**

Analytical outputs are also prepared for export and visualization.


In [545]:
customers = (
    clean_df[
        [
            "Customer Id",
            "Customer Fname",
            "Customer Lname",
            "Customer Segment",
            "Customer City",
            "Customer State",
            "Customer Country",
            "Customer Zipcode"
        ]
    ]
    .drop_duplicates(subset="Customer Id")
    .copy())


In [546]:
products = (
    clean_df[
        [
            "Product Card Id",
            "Product Name",
            "Category Name",
            "Department Name",
            "Product Price"
        ]
    ]
    .drop_duplicates(subset="Product Card Id")
    .copy())


In [547]:
orders = (
    clean_df[
        [
            "Order Id",
            "Customer Id",
            "order date (DateOrders)",
            "shipping date (DateOrders)",
            "Shipping Mode",
            "Days for shipping (real)",
            "Days for shipment (scheduled)",
            "Delivery Status",
            "Late_delivery_risk",
            "Market",
            "Order Region",
            "Order City",
            "Order State",
            "Order Country"
        ]
    ]
    .drop_duplicates(subset="Order Id")
    .copy())


In [548]:

order_items = (
    clean_df[
        [
            "Order Item Id",
            "Order Id",
            "Product Card Id",
            "Order Item Quantity",
            "Sales",
            "Order Item Product Price",
            "Order Item Discount",
            "Order Item Discount Rate",
            "Order Item Total",
            "Benefit per order",
            "Order Item Profit Ratio"
        ]
    ]
    .copy())

In [549]:
print("Customers:", len(customers))
print("Products:", len(products))
print("Orders:", len(orders))
print("Order Items:", len(order_items))

Customers: 20652
Products: 118
Orders: 65752
Order Items: 180519


In [550]:
customers.to_csv(
    "../Datasets/processed/customers.csv",
    index=False
)

products.to_csv(
    "../Datasets/processed/products.csv",
    index=False
)

orders.to_csv(
    "../Datasets/processed/orders.csv",
    index=False
)

order_items.to_csv(
    "../Datasets/processed/order_items.csv",
    index=False
)

In [551]:
forecast_df.to_csv(
    "../Datasets/processed/product_365_forecast.csv",
    index=False
)

actual_forecast_df.to_csv(
    "../Datasets/processed/product_365_actual_forecast.csv",
    index=False
)
forecast_model_comparison.to_csv(
    "../Datasets/processed/forecast_model_comparison.csv",
    index=False
)
inventory_planning.to_csv(
    "../Datasets/processed/product_365_inventory_planning.csv",
    index=False
)
priority_summary.to_csv(
    "../Datasets/processed/inventory_priority_summary.csv",
    index=False
)
product_abc.to_csv(
    "../Datasets/processed/product_abc.csv",
    index=False
)